# Day G5 — 단백질: 구조 예측과 단백질 언어 모델

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/G-ai-biomed/day-G5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/G-ai-biomed/day-G5.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q fair-esm

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — MSA로 가린 아미노산 맞히기, k-mer 벡터

In [ ]:
import numpy as np
from collections import Counter

rng = np.random.default_rng(0)
# (1) '빈칸 맞히기' 직관: 같은 가족 서열(가상 MSA)에서 가려진 아미노산을 예측한다
msa = ["MKVLAAGIVGLLAQ", "MKVLSAGIVGLLAQ", "MRVLAAGLVGLIAQ", "MKILAAGIVGMLSQ",
       "MKVLAAGVVGLLAE", "MKVMAAGIVGLLAQ", "MRVLAAGIIGLLAQ", "MKVLGAGIVGLVAQ"]
L = len(msa[0])

def predict_masked(seqs, pos):
    """다른 서열들의 같은 자리에서 가장 흔한 글자 = 가장 단순한 '언어 모델'."""
    return Counter(s[pos] for s in seqs).most_common(1)[0][0]

hits = total = 0
for i, s in enumerate(msa):                      # 한 서열씩 빼 두고(leave-one-out) 자리마다 가린다
    others = msa[:i] + msa[i + 1:]
    for pos in range(L):
        hits += predict_masked(others, pos) == s[pos]
        total += 1
print(f"가족 정보로 가린 글자 맞히기: {hits}/{total} = {hits / total:.0%}")
print(f"무작위로 20종 중 하나 찍기의 기대값: {1 / 20:.0%}")

# 자리별 보존도(엔트로피가 낮을수록 보존): 언어 모델이 '중요한 자리'를 배우는 단서
ent = []
for pos in range(L):
    p = np.array(list(Counter(s[pos] for s in msa).values())) / len(msa)
    ent.append(abs(-(p * np.log2(p)).sum()))   # abs: -0.0 표시 방지
print("자리별 엔트로피:", np.round(ent, 2))

# (2) 고전적 서열 표현: 2-mer 빈도 벡터 (단백질 언어 모델 임베딩 이전의 방식)
from itertools import product
AA = "ACDEFGHIKLMNPQRSTVWY"
kmers = ["".join(p) for p in product(AA, repeat=2)]
def kmer_vec(s):
    c = Counter(s[i:i + 2] for i in range(len(s) - 1))
    return np.array([c[k] for k in kmers], dtype=float)
v = kmer_vec(msa[0])
print("2-mer 벡터 길이:", len(v), "| 0이 아닌 칸:", int((v > 0).sum()))

### 예시 2 — 예측 구조 파일에서 pLDDT 읽기

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
# 가상 예측 구조 파일(PDB 형식의 CA 원자 줄만). AlphaFold 계열 파일은 B-factor 칸(61–66열)에 pLDDT를 넣는다.
# 1–60번은 잘 접힌 도메인(높은 pLDDT), 61–100번은 무질서 꼬리(낮은 pLDDT)를 흉내 낸 값이다.
plddt_true = np.r_[rng.normal(91, 3, 60), rng.normal(42, 8, 40)].clip(20, 99)
lines = []
for i, b in enumerate(plddt_true, start=1):
    lines.append(f"ATOM  {i:5d}  CA  ALA A{i:4d}    {0.0:8.3f}{0.0:8.3f}{0.0:8.3f}{1.0:6.2f}{b:6.2f}           C")
pdb_text = "\n".join(lines)

# 읽기: 고정 폭 열에서 잔기 번호와 B-factor(=pLDDT)를 꺼낸다
res, plddt = [], []
for line in pdb_text.splitlines():
    if line.startswith("ATOM") and line[12:16].strip() == "CA":
        res.append(int(line[22:26])); plddt.append(float(line[60:66]))
plddt = np.array(plddt)

bands = [(90, 101, "매우 높음(>90)"), (70, 90, "높음(70–90)"), (50, 70, "낮음(50–70)"), (0, 50, "매우 낮음(<50)")]
for lo, hi, name in bands:
    print(f"{name:<14} 잔기 {((plddt >= lo) & (plddt < hi)).sum():>3}개")

# 낮은 신뢰도 구간을 이어진 범위로 찾기: 잔기마다 들쭉날쭉하므로 앞뒤 2잔기씩(창 5) 평균한 뒤 자른다
smooth = np.convolve(plddt, np.ones(5) / 5, mode="same")
low = smooth < 50
starts = [res[i] for i in range(len(res)) if low[i] and (i == 0 or not low[i - 1])]
ends = [res[i] for i in range(len(res)) if low[i] and (i == len(res) - 1 or not low[i + 1])]
print("pLDDT<50 연속 구간(5잔기 이상):", [(s, e) for s, e in zip(starts, ends) if e - s >= 4])

### 예시 3 — ESM-2로 단백질 서열을 벡터로 바꾸기

ESM-2는 두 가지 방법으로 받을 수 있습니다. 공식 패키지 `fair-esm`은 Meta 서버에서 가중치를 받는데, **2026년 11월 11일에 그 서버(`dl.fbaipublicfiles.com`)의 보안 인증서가 만료되어** 이 레슨을 쓴 시점에는 `CERTIFICATE_VERIFY_FAILED` 오류로 내려받기가 실패했습니다. 인증서 검증을 끄고 받는 것은 안전하지 않으므로 하지 않습니다. 같은 모델이 Hugging Face에도 공식 게시되어 있어, 아래에서는 `transformers`로 그쪽에서 받습니다.

In [ ]:
# 필요: pip install transformers (PyTorch는 G2 예시 3 참고). 첫 실행 때 모델 약 30 MB를 내려받는다.
import torch
from transformers import AutoTokenizer, EsmModel
name = "facebook/esm2_t6_8M_UR50D"                        # 가장 작은 ESM-2
tok = AutoTokenizer.from_pretrained(name)
model = EsmModel.from_pretrained(name, add_pooling_layer=False).eval()
tokens = tok(["MKVLAAGIVGLLAQ"], return_tensors="pt")
with torch.no_grad():
    out = model(**tokens)
emb = out.last_hidden_state[0, 1:-1].mean(0)             # 시작·끝 토큰을 빼고 평균: 서열 하나 → 벡터 하나
print(tokens["input_ids"].shape, emb.shape)
print(emb[:5].round(decimals=3))

공식 `fair-esm` 패키지로 쓸 때의 형태는 아래와 같습니다. 서버 인증서가 갱신되면 그대로 동작해야 하지만, 위 이유로 이 레슨을 쓸 때는 **실행 확인 안 됨(가중치 서버 인증서 만료)** 입니다.

In [ ]:
import torch, esm
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch = alphabet.get_batch_converter()
_, _, tokens = batch([("demo", "MKVLAAGIVGLLAQ")])
with torch.no_grad():
    out = model(tokens, repr_layers=[6])
emb = out["representations"][6][0, 1:-1].mean(0)